# 04 - Modeling

This notebook trains, evaluates, and compares baseline and behavior-aware machine learning models for customer behavior shift detection.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, average_precision_score)
from sklearn.metrics import confusion_matrix, classification_report


In [14]:
# Load the modeling dataset
file_path = "../data/processed/behavior_change_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

Dataset shape: (19651, 32)
Columns:
['Customer ID', 'Month', 'transaction_count', 'total_quantity', 'total_spending', 'average_transaction_value', 'unique_products', 'previous_month', 'months_since_previous', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'change_transaction_count', 'change_total_quantity', 'change_total_spending', 'change_average_transaction_value', 'change_unique_products', 'pct_change_transaction_count', 'pct_change_total_quantity', 'pct_change_total_spending', 'pct_change_average_transaction_value', 'pct_change_unique_products', 'large_change_count', 'behavior_shift_candidate', 'core_large_change_count', 'core_behavior_shift_candidate', 'behavior_shift', 'historical_active_months', 'historical_transactions', 'historical_spending']


,Customer ID,Month,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products,previous_month,months_since_previous,previous_transaction_count,...,pct_change_average_transaction_value,pct_change_unique_products,large_change_count,behavior_shift_candidate,core_large_change_count,core_behavior_shift_candidate,behavior_shift,historical_active_months,historical_transactions,historical_spending
0,12346.0,2010-06,1,19,142.31,7.490000,19,2010-03,3,1.0,...,3.844732e+01,280.000000,3,True,3,True,1,0,1.0,27.05
1,12346.0,2011-01,1,74215,77183.60,77183.600000,1,2010-06,7,1.0,...,1.030389e+06,-94.736842,3,True,3,True,1,1,2.0,169.36
2,12347.0,2010-12,1,319,711.79,22.960968,31,2010-10,2,1.0,...,5.018702e+01,-22.500000,0,False,0,False,0,0,1.0,611.53
3,12347.0,2011-01,1,315,475.39,16.392759,29,2010-12,1,1.0,...,-2.860598e+01,-6.451613,0,False,0,False,0,1,2.0,1323.32
4,12347.0,2011-04,1,483,636.25,26.510417,24,2011-01,3,1.0,...,6.172029e+01,-17.241379,0,False,0,False,0,2,3.0,1798.71


In [15]:
# Define target
target_column = "behavior_shift"

# Baseline features
baseline_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending"
]

# Behavior-aware features
behavior_aware_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending",
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products",
    "months_since_previous"
]

print("Target:", target_column)

print("\nBaseline features:")
print(baseline_features)

print("\nBehavior-aware features:")
print(behavior_aware_features)

Target: behavior_shift

Baseline features:
['historical_active_months', 'historical_transactions', 'historical_spending']

Behavior-aware features:
['historical_active_months', 'historical_transactions', 'historical_spending', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'months_since_previous']


In [16]:
# Define time-based split boundaries

train_end = "2011-05"
validation_end = "2011-08"

# Training set
train_df = df[
    df["Month"] <= train_end
].copy()

# Validation set
validation_df = df[
    (df["Month"] > train_end) &
    (df["Month"] <= validation_end)
].copy()

# Test set
test_df = df[
    df["Month"] > validation_end
].copy()

print("Train period:",
      train_df["Month"].min(), "to", train_df["Month"].max())

print("Validation period:",
      validation_df["Month"].min(), "to", validation_df["Month"].max())

print("Test period:",
      test_df["Month"].min(), "to", test_df["Month"].max())

print("\nRows:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train period: 2010-01 to 2011-05
Validation period: 2011-06 to 2011-08
Test period: 2011-09 to 2011-12

Rows:
Train: 12832
Validation: 2553
Test: 4266


In [24]:
# Check target distribution across all data splits

for name, dataset in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} target distribution:")
    
    print(dataset[target_column].value_counts().sort_index())
    
    print("\nPercentage:")
    print(
        dataset[target_column]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )


Train target distribution:
behavior_shift
0    10745
1     2087
Name: count, dtype: int64

Percentage:
behavior_shift
0    83.74
1    16.26
Name: proportion, dtype: float64

Validation target distribution:
behavior_shift
0    2198
1     355
Name: count, dtype: int64

Percentage:
behavior_shift
0    86.09
1    13.91
Name: proportion, dtype: float64

Test target distribution:
behavior_shift
0    3471
1     795
Name: count, dtype: int64

Percentage:
behavior_shift
0    81.36
1    18.64
Name: proportion, dtype: float64


In [18]:
# Prepare baseline features and target

X_train_baseline = train_df[baseline_features].copy()
X_validation_baseline = validation_df[baseline_features].copy()
X_test_baseline = test_df[baseline_features].copy()

y_train = train_df[target_column].copy()
y_validation = validation_df[target_column].copy()
y_test = test_df[target_column].copy()

print("Baseline training shape:", X_train_baseline.shape)
print("Baseline validation shape:", X_validation_baseline.shape)
print("Baseline test shape:", X_test_baseline.shape)

print("\nTarget shapes:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

Baseline training shape: (12832, 3)
Baseline validation shape: (2553, 3)
Baseline test shape: (4266, 3)

Target shapes:
Train: (12832,)
Validation: (2553,)
Test: (4266,)


In [19]:
# Define baseline preprocessing

baseline_log_features = [
    "historical_transactions",
    "historical_spending"
]

baseline_numeric_features = [
    "historical_active_months"
]

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "log",
            Pipeline([
                ("log1p", FunctionTransformer(np.log1p)),
                ("scaler", StandardScaler())
            ]),
            baseline_log_features
        ),
        (
            "numeric",
            StandardScaler(),
            baseline_numeric_features
        )
    ]
)

print("Baseline preprocessing pipeline created.")

Baseline preprocessing pipeline created.


In [20]:
# Build the baseline Logistic Regression pipeline

baseline_model = Pipeline([
    ("preprocessing", baseline_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline Logistic Regression pipeline created.")

Baseline Logistic Regression pipeline created.


In [21]:
# Fit the baseline model on the training data

baseline_model.fit(
    X_train_baseline,
    y_train
)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [22]:
# Generate predictions on the validation set

y_validation_pred = baseline_model.predict(
    X_validation_baseline
)

y_validation_proba = baseline_model.predict_proba(
    X_validation_baseline
)[:, 1]

print("Validation predictions generated.")
print("Predictions shape:", y_validation_pred.shape)
print("Probabilities shape:", y_validation_proba.shape)

Validation predictions generated.
Predictions shape: (2553,)
Probabilities shape: (2553,)


In [25]:
baseline_validation_metrics = {
    "Precision": precision_score(y_validation, y_validation_pred),
    "Recall": recall_score(y_validation, y_validation_pred),
    "F1": f1_score(y_validation, y_validation_pred),
    "ROC-AUC": roc_auc_score(y_validation, y_validation_proba),
    "PR-AUC": average_precision_score(
        y_validation,
        y_validation_proba
    )
}

print("Baseline Validation Metrics:")
for metric, value in baseline_validation_metrics.items():
    print(f"{metric}: {value:.4f}")

Baseline Validation Metrics:
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
ROC-AUC: 0.5763
PR-AUC: 0.1804


## Baseline Validation Probability Analysis

The predicted probabilities are inspected to understand how the baseline model
separates the two target classes.

We compare the probability distribution overall and the average predicted
probability for each actual class.

In [27]:
# Inspect the distribution of predicted probabilities and compare them by actual class

print("Validation probability summary:")

print(pd.Series(y_validation_proba).describe())

print("\nMean probability by actual class:")

print(
    pd.DataFrame({
        "target": y_validation.values,
        "probability": y_validation_proba
    })
    .groupby("target")["probability"]
    .agg(["count", "mean", "median", "min", "max"])
)

Validation probability summary:
count    2553.000000
mean        0.182089
std         0.054922
min         0.040110
25%         0.144876
50%         0.175919
75%         0.213064
max         0.572563
dtype: float64

Mean probability by actual class:
        count      mean    median       min       max
target                                               
0        2198  0.180073  0.174376  0.040110  0.572563
1         355  0.194573  0.194415  0.053993  0.456900


## Baseline Threshold Analysis

The default classification threshold of 0.5 produces no positive predictions,
resulting in zero recall and F1-score.

Since the predicted probabilities show some separation between the two classes,
we evaluate alternative decision thresholds on the validation set.

The threshold will be selected using the F1-score on the validation data and
then kept fixed for the final test evaluation.

## Baseline Threshold Analysis

The default threshold of 0.5 produces no positive predictions.
Alternative thresholds are evaluated on the validation set to determine
whether a lower threshold improves the model's ability to detect behavioral shifts.

F1-score is used as the primary criterion because the target classes are imbalanced
and both precision and recall are important.

In [36]:
# Evaluate baseline performance across different classification thresholds

thresholds = np.arange(0.10, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    y_validation_pred_threshold = (
        y_validation_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            y_validation_pred_threshold,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            y_validation_pred_threshold,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            y_validation_pred_threshold,
            zero_division=0
        )
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,threshold,precision,recall,f1
0,0.10,0.140708,0.974648,0.245913
1,0.15,0.151432,0.774648,0.253339
2,0.20,0.188746,0.453521,0.266556
3,0.25,0.212000,0.149296,0.175207
4,0.30,0.225352,0.045070,0.075117
5,0.35,0.320000,0.022535,0.042105
6,0.40,0.166667,0.002817,0.005540
7,0.45,0.200000,0.002817,0.005556
8,0.50,0.000000,0.000000,0.000000


### Select the Best Validation Threshold

The threshold that achieves the highest F1-score on the validation set is selected
for the final baseline evaluation.

In [37]:
# Select the threshold with the highest validation F1-score

best_threshold_row = threshold_results_df.loc[
    threshold_results_df["f1"].idxmax()
]

best_threshold = best_threshold_row["threshold"]

print("Best threshold:", best_threshold)
print("Precision:", round(best_threshold_row["precision"], 4))
print("Recall:", round(best_threshold_row["recall"], 4))
print("F1:", round(best_threshold_row["f1"], 4))

Best threshold: 0.20000000000000004
Precision: 0.1887
Recall: 0.4535
F1: 0.2666


## Final Baseline Validation Evaluation

The threshold selected using validation F1-score is applied to generate
the final baseline validation predictions.

This threshold will be kept fixed when evaluating the baseline model
on the test set.

In [38]:
# Apply the selected threshold to generate final baseline validation predictions

baseline_validation_threshold = best_threshold

y_validation_pred_baseline = (
    y_validation_proba >= baseline_validation_threshold
).astype(int)

print("Baseline validation threshold:", baseline_validation_threshold)
print("Positive predictions:", y_validation_pred_baseline.sum())
print("Total validation observations:", len(y_validation_pred_baseline))

Baseline validation threshold: 0.20000000000000004
Positive predictions: 853
Total validation observations: 2553


## Baseline Confusion Matrix

The confusion matrix shows how the baseline model's validation predictions
compare with the actual behavior-shift labels at the selected threshold.

In [39]:
# Evaluate baseline validation predictions using a confusion matrix


baseline_cm = confusion_matrix(
    y_validation,
    y_validation_pred_baseline
)

print("Baseline Validation Confusion Matrix:")
print(baseline_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_baseline,
        target_names=["No Shift", "Behavior Shift"],
        zero_division=0
    )
)

Baseline Validation Confusion Matrix:
[[1506  692]
 [ 194  161]]

Classification Report:
                precision    recall  f1-score   support

      No Shift       0.89      0.69      0.77      2198
Behavior Shift       0.19      0.45      0.27       355

      accuracy                           0.65      2553
     macro avg       0.54      0.57      0.52      2553
  weighted avg       0.79      0.65      0.70      2553



## Baseline Validation Results

Using the threshold selected from validation F1-score (0.20), the baseline
Logistic Regression achieved a recall of 0.45 and an F1-score of 0.27
for the Behavior Shift class.

The model correctly identified 161 of 355 behavior-shift observations,
while producing 692 false positives.

These results provide the baseline against which the behavior-aware model
will be compared.

## Behavior-Aware Model

The behavior-aware model extends the baseline feature set with recent
customer behavioral information from the previous active month.

The objective is to determine whether incorporating recent behavioral
features improves behavior shift detection compared with the baseline model.

## Behavior-Aware Model

The behavior-aware model extends the baseline feature set with recent
customer behavioral information from the previous active month.

The objective is to determine whether incorporating recent behavioral
features improves behavior shift detection compared with the baseline model.

In [41]:
# Prepare behavior-aware features and target

X_train_behavior = train_df[behavior_aware_features].copy()
X_validation_behavior = validation_df[behavior_aware_features].copy()
X_test_behavior = test_df[behavior_aware_features].copy()

print("Behavior-aware training shape:", X_train_behavior.shape)
print("Behavior-aware validation shape:", X_validation_behavior.shape)
print("Behavior-aware test shape:", X_test_behavior.shape)

print("\nTarget shapes:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

Behavior-aware training shape: (12832, 9)
Behavior-aware validation shape: (2553, 9)
Behavior-aware test shape: (4266, 9)

Target shapes:
Train: (12832,)
Validation: (2553,)
Test: (4266,)


### Behavior-Aware Preprocessing

Highly right-skewed behavioral features are log-transformed using log1p
before standardization.

The preprocessing is fitted only on the training data through the pipeline
to avoid data leakage.

In [42]:
# Define behavior-aware preprocessing

behavior_aware_log_features = [
    "historical_transactions",
    "historical_spending",
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products"
]

behavior_aware_numeric_features = [
    "historical_active_months",
    "months_since_previous"
]

behavior_aware_preprocessor = ColumnTransformer(
    transformers=[
        (
            "log",
            Pipeline([
                ("log1p", FunctionTransformer(np.log1p)),
                ("scaler", StandardScaler())
            ]),
            behavior_aware_log_features
        ),
        (
            "numeric",
            StandardScaler(),
            behavior_aware_numeric_features
        )
    ]
)

print("Behavior-aware preprocessing pipeline created.")

Behavior-aware preprocessing pipeline created.


### Behavior-Aware Logistic Regression

A Logistic Regression classifier is used to provide a direct comparison
with the baseline model.

The same model family is intentionally used so that any performance
difference can be attributed primarily to the additional behavioral features.

In [43]:
# Create the behavior-aware Logistic Regression pipeline

behavior_aware_model = Pipeline([
    ("preprocessor", behavior_aware_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Behavior-aware Logistic Regression pipeline created.")

Behavior-aware Logistic Regression pipeline created.


### Generate Behavior-Aware Validation Predictions

Generate validation-set predictions and probabilities using the behavior-aware Logistic Regression model.

In [45]:
# Generate behavior-aware validation predictions

behavior_aware_model.fit(
    X_train_behavior,
    y_train
)

y_validation_behavior_aware_pred = (
    behavior_aware_model.predict(
        X_validation_behavior
    )
)

y_validation_behavior_aware_proba = (
    behavior_aware_model.predict_proba(
        X_validation_behavior
    )[:, 1]
)

print("Behavior-aware validation predictions generated.")
print(
    "Predictions shape:",
    y_validation_behavior_aware_pred.shape
)
print(
    "Probabilities shape:",
    y_validation_behavior_aware_proba.shape
)

Behavior-aware validation predictions generated.
Predictions shape: (2553,)
Probabilities shape: (2553,)


## Behavior-Aware Validation Evaluation

The behavior-aware model is first evaluated using the default classification
threshold of 0.5.

This provides a direct comparison with the baseline model before optimizing
the decision threshold.

In [46]:
# Evaluate behavior-aware model using the default threshold of 0.5


behavior_aware_precision = precision_score(
    y_validation,
    y_validation_behavior_aware_pred,
    zero_division=0
)

behavior_aware_recall = recall_score(
    y_validation,
    y_validation_behavior_aware_pred,
    zero_division=0
)

behavior_aware_f1 = f1_score(
    y_validation,
    y_validation_behavior_aware_pred,
    zero_division=0
)

behavior_aware_roc_auc = roc_auc_score(
    y_validation,
    y_validation_behavior_aware_proba
)

behavior_aware_pr_auc = average_precision_score(
    y_validation,
    y_validation_behavior_aware_proba
)

print("Behavior-Aware Validation Metrics:")
print(f"Precision: {behavior_aware_precision:.4f}")
print(f"Recall: {behavior_aware_recall:.4f}")
print(f"F1: {behavior_aware_f1:.4f}")
print(f"ROC-AUC: {behavior_aware_roc_auc:.4f}")
print(f"PR-AUC: {behavior_aware_pr_auc:.4f}")

Behavior-Aware Validation Metrics:
Precision: 0.5529
Recall: 0.2648
F1: 0.3581
ROC-AUC: 0.7794
PR-AUC: 0.4347


## Behavior-Aware Probability Analysis

The predicted probabilities are inspected to understand how the
behavior-aware model separates the two target classes before selecting
an operating threshold.

In [47]:
# Inspect behavior-aware validation probability distribution

print("Validation probability summary:")

print(
    pd.Series(
        y_validation_behavior_aware_proba
    ).describe()
)

print("\nMean probability by actual class:")

print(
    pd.DataFrame({
        "target": y_validation.values,
        "probability": y_validation_behavior_aware_proba
    })
    .groupby("target")["probability"]
    .agg(["count", "mean", "median", "min", "max"])
)

Validation probability summary:
count    2553.000000
mean        0.188913
std         0.176227
min         0.000943
25%         0.068775
50%         0.133440
75%         0.245366
max         0.998967
dtype: float64

Mean probability by actual class:
        count      mean    median       min       max
target                                               
0        2198  0.159653  0.119792  0.000943  0.961315
1         355  0.370075  0.304917  0.004068  0.998967


## Behavior-Aware Threshold Analysis

The classification threshold is varied on the validation set to identify
the threshold that provides the best F1-score for the Behavior Shift class.

The selected threshold will be fixed before evaluating the model on the
test set.

In [48]:
# Evaluate behavior-aware model across different classification thresholds

thresholds = np.arange(0.10, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred = (
        y_validation_behavior_aware_proba >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(threshold_results_df)

   threshold  precision    recall        f1
0       0.10   0.195734  0.878873  0.320164
1       0.15   0.238428  0.769014  0.364000
2       0.20   0.294686  0.687324  0.412511
3       0.25   0.347200  0.611268  0.442857
4       0.30   0.393548  0.515493  0.446341
5       0.35   0.432361  0.459155  0.445355
6       0.40   0.479452  0.394366  0.432767
7       0.45   0.520179  0.326761  0.401384
8       0.50   0.552941  0.264789  0.358095


### Select the Best Behavior-Aware Threshold

The threshold with the highest validation F1-score is selected as the
operating threshold for the behavior-aware model.

In [49]:
# Select the threshold with the highest validation F1-score

best_behavior_aware_row = (
    threshold_results_df
    .loc[threshold_results_df["f1"].idxmax()]
)

best_behavior_aware_threshold = (
    best_behavior_aware_row["threshold"]
)

print(
    f"Best threshold: "
    f"{best_behavior_aware_threshold:.2f}"
)

print(
    f"Precision: "
    f"{best_behavior_aware_row['precision']:.4f}"
)

print(
    f"Recall: "
    f"{best_behavior_aware_row['recall']:.4f}"
)

print(
    f"F1: "
    f"{best_behavior_aware_row['f1']:.4f}"
)

Best threshold: 0.30
Precision: 0.3935
Recall: 0.5155
F1: 0.4463


## Selected Behavior-Aware Threshold

A threshold of 0.30 was selected based on the highest validation F1-score.

At this threshold, the behavior-aware model achieved a precision of 0.39,
a recall of 0.52, and an F1-score of 0.45 for the Behavior Shift class.

The threshold is selected using validation data and will remain fixed for
the final test evaluation.

In [50]:
# Fix the selected behavior-aware classification threshold

behavior_aware_threshold = best_behavior_aware_threshold

y_validation_behavior_aware_pred_threshold = (
    y_validation_behavior_aware_proba >= behavior_aware_threshold
).astype(int)

print(
    "Behavior-aware validation threshold:",
    behavior_aware_threshold
)

print(
    "Positive predictions:",
    y_validation_behavior_aware_pred_threshold.sum()
)

print(
    "Total validation observations:",
    len(y_validation_behavior_aware_pred_threshold)
)

Behavior-aware validation threshold: 0.30000000000000004
Positive predictions: 465
Total validation observations: 2553


## Behavior-Aware Validation Confusion Matrix

The confusion matrix and classification report are used to evaluate the
behavior-aware model at the selected validation threshold of 0.30.

In [52]:
# Evaluate behavior-aware predictions at the selected threshold

behavior_aware_cm = confusion_matrix(
    y_validation,
    y_validation_behavior_aware_pred_threshold
)

print("Behavior-Aware Validation Confusion Matrix:")
print(behavior_aware_cm)

print("\nClassification Report:")

print(
    classification_report(
        y_validation,
        y_validation_behavior_aware_pred_threshold,
        target_names=["No Shift", "Behavior Shift"],
        zero_division=0
    )
)

Behavior-Aware Validation Confusion Matrix:
[[1916  282]
 [ 172  183]]

Classification Report:
                precision    recall  f1-score   support

      No Shift       0.92      0.87      0.89      2198
Behavior Shift       0.39      0.52      0.45       355

      accuracy                           0.82      2553
     macro avg       0.66      0.69      0.67      2553
  weighted avg       0.84      0.82      0.83      2553



## Baseline vs Behavior-Aware Validation Comparison

The baseline and behavior-aware models are compared on the same validation
set using Precision, Recall, F1-score, ROC-AUC, and PR-AUC.

The comparison evaluates whether incorporating previous behavioral information
improves the detection of customer behavior shifts.

In [54]:
# Inspect validation prediction variables

[name for name in globals() if "validation" in name.lower() and "pred" in name.lower()]

['y_validation_pred',
 'y_validation_pred_threshold',
 'y_validation_pred_baseline',
 'y_validation_behavior_aware_pred',
 'y_validation_behavior_aware_pred_threshold']

In [58]:
# Compare baseline and behavior-aware validation performance

comparison_results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Behavior-Aware"
    ],
    "Threshold": [
        0.20,
        0.30
    ],
    "Precision": [
        precision_score(
            y_validation,
            y_validation_pred_baseline,
            zero_division=0
        ),
        precision_score(
            y_validation,
            y_validation_behavior_aware_pred_threshold,
            zero_division=0
        )
    ],
    "Recall": [
        recall_score(
            y_validation,
            y_validation_pred_baseline,
            zero_division=0
        ),
        recall_score(
            y_validation,
            y_validation_behavior_aware_pred_threshold,
            zero_division=0
        )
    ],
    "F1": [
        f1_score(
            y_validation,
            y_validation_pred_baseline,
            zero_division=0
        ),
        f1_score(
            y_validation,
            y_validation_behavior_aware_pred_threshold,
            zero_division=0
        )
    ],
    "ROC-AUC": [
        roc_auc_score(
            y_validation,
            y_validation_proba
        ),
        roc_auc_score(
            y_validation,
            y_validation_behavior_aware_proba
        )
    ],
    "PR-AUC": [
        average_precision_score(
            y_validation,
            y_validation_proba
        ),
        average_precision_score(
            y_validation,
            y_validation_behavior_aware_proba
        )
    ]
})

comparison_results

,Model,Threshold,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Baseline,0.2,0.188746,0.453521,0.266556,0.576253,0.180441
1,Behavior-Aware,0.3,0.393548,0.515493,0.446341,0.779406,0.434740


## Behavior-Aware Test Evaluation

The behavior-aware model is evaluated on the held-out test set using the
threshold selected on the validation set.

The test set is not used for threshold selection or model tuning, providing
an unbiased estimate of final model performance.

In [61]:
# Prepare behavior-aware features and target for the test evaluation

X_train_behavior_aware = train_df[behavior_aware_features].copy()
X_validation_behavior_aware = validation_df[behavior_aware_features].copy()
X_test_behavior_aware = test_df[behavior_aware_features].copy()

y_train = train_df[target_column].copy()
y_validation = validation_df[target_column].copy()
y_test = test_df[target_column].copy()

print("Behavior-aware training shape:", X_train_behavior_aware.shape)
print("Behavior-aware validation shape:", X_validation_behavior_aware.shape)
print("Behavior-aware test shape:", X_test_behavior_aware.shape)

print("\nTarget shapes:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

Behavior-aware training shape: (12832, 9)
Behavior-aware validation shape: (2553, 9)
Behavior-aware test shape: (4266, 9)

Target shapes:
Train: (12832,)
Validation: (2553,)
Test: (4266,)


In [62]:
# Generate behavior-aware test predictions

behavior_aware_model.fit(
    X_train_behavior_aware,
    y_train
)

y_test_behavior_aware_proba = behavior_aware_model.predict_proba(
    X_test_behavior_aware
)[:, 1]

y_test_behavior_aware_pred = (
    y_test_behavior_aware_proba >= behavior_aware_threshold
).astype(int)

print("Behavior-aware test predictions generated.")
print("Predictions shape:", y_test_behavior_aware_pred.shape)
print("Probabilities shape:", y_test_behavior_aware_proba.shape)

Behavior-aware test predictions generated.
Predictions shape: (4266,)
Probabilities shape: (4266,)


## Behavior-Aware Test Confusion Matrix

The confusion matrix and classification report provide a detailed evaluation
of the final behavior-aware model on the held-out test set using the threshold
selected during validation.

In [63]:
# Evaluate behavior-aware model on the test set

test_precision = precision_score(
    y_test,
    y_test_behavior_aware_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_behavior_aware_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_behavior_aware_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_behavior_aware_proba
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_behavior_aware_proba
)

print("Behavior-Aware Test Metrics:")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1: {test_f1:.4f}")
print(f"ROC-AUC: {test_roc_auc:.4f}")
print(f"PR-AUC: {test_pr_auc:.4f}")

Behavior-Aware Test Metrics:
Precision: 0.4395
Recall: 0.4201
F1: 0.4296
ROC-AUC: 0.7474
PR-AUC: 0.4494


## Behavior-Aware Test Confusion Matrix

The confusion matrix and classification report provide a detailed evaluation
of the final behavior-aware model on the held-out test set using the threshold
selected during validation.

In [64]:
# Evaluate the final behavior-aware model on the test set

test_confusion_matrix = confusion_matrix(
    y_test,
    y_test_behavior_aware_pred
)

test_classification_report = classification_report(
    y_test,
    y_test_behavior_aware_pred,
    target_names=["No Shift", "Behavior Shift"],
    zero_division=0
)

print("Behavior-Aware Test Confusion Matrix:")
print(test_confusion_matrix)

print("\nClassification Report:")
print(test_classification_report)

Behavior-Aware Test Confusion Matrix:
[[3045  426]
 [ 461  334]]

Classification Report:
                precision    recall  f1-score   support

      No Shift       0.87      0.88      0.87      3471
Behavior Shift       0.44      0.42      0.43       795

      accuracy                           0.79      4266
     macro avg       0.65      0.65      0.65      4266
  weighted avg       0.79      0.79      0.79      4266

